# Digital technology and productivity
**Econometrics II · Units 1 to 3 · Class notebook**

The Ministry of Industry has budgeted a subsidy for digital technology in manufacturing firms. We are the consultants, and the ministry asks two things:

1. By how much does productivity rise when a firm has more digital capital per worker?
2. What would a **grant** do?

We work with a random sample of 2,000 firms. Both key variables are in logs: log value added per worker (`ln_prod`) and log digital capital per worker (`ln_digital`). The coefficient on `ln_digital` is therefore an **elasticity**.

*All firms, policy details and data are simulated.*

### Roadmap

| Part | Topic | Unit | What we learn |
|:---|:---|:---:|:---|
| 1 | The sample and the data | 1 | Cross-section, random sampling, OLS slope, bootstrap |
| 2 | Measurement error | 2 | Error in $y$ costs precision; error in $x$ causes attenuation |
| 3 | Building the model | 2 | Controls, $R^2$ vs adjusted $R^2$, quadratics, interactions, overfitting, rank |
| 4 | Omitted variable bias | 2 | Short vs long regression, the OVB formula |
| 5 | Instrumental variables | 3 | Relevance, exclusion, 2SLS, weak and invalid instruments |

### How to use this notebook

- **Kernel:** R (IRkernel, or the R runtime in Google Colab). Run the cells **from top to bottom**; later parts reuse objects created earlier.
- **Data:** the two CSV files are read directly from the course website on GitHub (`henryredondo.github.io/datasets`), so nothing needs to be downloaded or uploaded. An internet connection is required.
- **Standard errors:** always heteroskedasticity-robust **HC2**, the default of `lm_robust()` and `iv_robust()`. We only write `se_type` when we want something else (`"classical"`).
- **Tables:** standard errors appear in parentheses under each coefficient.

## Setup

**On Google Colab**, the next cell installs the packages the first time (1–2 minutes). On your own computer it does nothing; there, install them once with `install.packages(c("broom", "estimatr", "modelsummary", "tidyverse"))`.

In [ ]:
# Colab only: installs the packages this notebook needs (1-2 minutes the first time).
# On your own computer this cell does nothing.
need <- c("broom", "estimatr", "modelsummary", "tables", "tidyverse")
if (Sys.getenv("COLAB_RELEASE_TAG") != "" || dir.exists("/content")) {
  os <- readLines("/etc/os-release")
  codename <- sub(".*=", "", grep("^(UBUNTU|VERSION)_CODENAME=", os, value = TRUE)[1])
  if (is.na(codename)) codename <- "jammy"
  # binaries from Posit Package Manager (no compilation); they need R's user agent
  options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)),
          HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
                                  paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
  #missing <- setdiff(need, rownames(installed.packages()))
  #if (length(missing) > 0) install.packages(missing, Ncpus = 2)
}
missing <- setdiff(need, rownames(installed.packages()))
if (length(missing) > 0) install.packages(missing, Ncpus = 2)


In [ ]:
library(tidyverse)     # Read and manipulate data; make plots
library(broom)         # tidy(): coefficients; glance(): model statistics
library(estimatr)      # lm_robust(), iv_robust(): OLS and 2SLS with robust SEs
library(modelsummary)  # Regression tables

options(digits = 4)                                # Fewer decimals in printed output
options(repr.plot.width = 7, repr.plot.height = 4) # Size of plots in the notebook
options(modelsummary_output = "jupyter")           # Show modelsummary tables in the notebook

Every regression table shows the coefficients with their standard errors in parentheses below, and the same three statistics at the bottom: $N$, $R^2$ and adjusted $R^2$. We list those statistics once and pass them to `modelsummary()` with `gof_map = gof_rows`.

In [ ]:
gof_rows <- tribble(
  ~raw,            ~clean,        ~fmt,
  "nobs",          "N",           0,
  "r.squared",     "R2",          4,
  "adj.r.squared", "Adjusted R2", 4
)

---
# Part 1. The sample and the data (Unit 1)

We receive the firm survey, `firms_survey.csv`. Productivity and digital capital are survey answers, so we do not yet know how reliable they are.

### 1.1 How many firms and variables? Is any firm repeated?

> How many firms and how many variables does the file contain? Check whether any firm appears twice.

In [ ]:
survey <- read_csv("https://raw.githubusercontent.com/henryredondo/henryredondo.github.io/main/datasets/firms_survey.csv",
                   show_col_types = FALSE)

We create the variables used throughout the case:

| New variable | Definition |
|:---|:---|
| `sector` | Factor; **food** is the reference sector |
| `ln_employees` | $\ln$(total workers) |
| `skilled_share` | skilled workers / total workers |
| `firm_age_sq` | firm age squared |
| `size_x_skill` | `ln_employees` × `skilled_share` (an interaction) |

In [ ]:
survey <- survey |>
  mutate(
    sector = factor(sector, levels = c("food", "textiles", "metals", "chemicals")),
    ln_employees = log(n_workers_total),
    skilled_share = n_workers_skilled / n_workers_total,
    firm_age_sq = firm_age^2,
    size_x_skill = ln_employees * skilled_share
  )

In [ ]:
glimpse(survey)

- `ln_prod_survey`: log value added per worker (thousand euros).
- `ln_digital_survey`: log digital capital per worker (thousand euros).

In [ ]:
summary(select(survey, n_workers_total, firm_age, skilled_share,
               ln_prod_survey, ln_digital_survey))

Does any `firm_id` appear more than once? An empty result means no.

In [ ]:
survey |>
  count(firm_id) |>
  filter(n > 1)

### 1.2 Why is this a cross-section?

> In two sentences, explain why this data is a *cross section*.

**Discussion.** Each firm appears once. Digital capital is measured in 2022–23 and productivity in 2024. We cannot follow the same firm over time or remove time-invariant unobserved firm characteristics by differencing.

### 1.3 Why is this a random sample?

> The firms are a simple random sample from the register. Name two things this guarantees and one thing it does not.

**Discussion.** The 2,000 firms are a simple random sample of manufacturing firms.
- Sample means are unbiased for population means.
- For inference we assume independent observations. That is an approximation if firms share local shocks or sampling is without replacement from a finite population.
- We observe **one** sample, not the full population.

### 1.4 The slope of the linear projection

> Compute the slope of `ln_prod_survey` on `ln_digital_survey` as $\mathrm{Cov}(x,y)/\mathrm{Var}(x)$ and check it against the regression. Which population quantity does it estimate? Does it have a causal meaning?

The population slope is $\beta = \mathrm{Cov}(x,y)/\mathrm{Var}(x)$. We replace population moments by sample moments.

In [ ]:
slope_by_hand <- cov(survey$ln_digital_survey, survey$ln_prod_survey) /
  var(survey$ln_digital_survey)
slope_by_hand

Now the same slope with `lm_robust()`. It reports heteroskedasticity-robust standard errors directly: HC2 by default, so we do not need to write `se_type`.

In [ ]:
simple_model <- lm_robust(ln_prod_survey ~ ln_digital_survey,
                          data = survey)

c(by_hand = slope_by_hand,
  with_lm_robust = coef(simple_model)[["ln_digital_survey"]])

In [ ]:
summary(simple_model)

### 1.5 Sampling variability: the bootstrap

> Draw 1,000 bootstrap samples (with replacement) and compute the slope in each. Compare the standard deviation of the 1,000 slopes with the robust standard error. What does the spread tell us? Why can we not simply draw new samples from the population?

**Recipe:** draw $N$ firms *with replacement*, compute the slope, repeat 1,000 times. The seed makes the random draws reproducible.

In [ ]:
set.seed(20261123)
number_of_draws <- 1000
slope_in_each_draw <- numeric(number_of_draws)

for (draw in 1:number_of_draws) {
  firms_drawn <- slice_sample(survey, prop = 1, replace = TRUE)
  slope_in_each_draw[draw] <-
    cov(firms_drawn$ln_digital_survey, firms_drawn$ln_prod_survey) /
    var(firms_drawn$ln_digital_survey)
}

The standard deviation of the bootstrap slopes estimates the standard error:

In [ ]:
tibble(
  bootstrap_se = sd(slope_in_each_draw),
  robust_se = simple_model$std.error[["ln_digital_survey"]]
)

In [ ]:
ggplot(tibble(slope = slope_in_each_draw), aes(x = slope)) +
  geom_histogram(bins = 30, fill = "grey85", colour = "grey40") +
  geom_vline(xintercept = slope_by_hand, linewidth = 0.8) +
  labs(title = "The slope in 1,000 bootstrap samples",
       x = "Estimated slope", y = "Count") +
  theme_minimal()

> **Takeaway.** The slope is an **association**, not yet a causal effect. Measurement error and omitted firm characteristics may affect it. Parts 2 and 4 deal with each.

---
# Part 2. Measurement error (Unit 2)

The survey contains **noisy** measures of productivity and digital capital. The administrative (tax and customs) records are assumed to measure both **without error**.

Every regression in this part controls for size, age and age squared, skilled share, size × skilled share, sector **and management quality**, so that omitted variable bias does not mix in. Write $y$ for productivity and $x$ for digital capital.

### 2.1 What the consultants initially observe: the survey

> Using only the survey, regress `ln_prod_survey` on `ln_digital_survey` and the controls. Report the coefficient, its robust standard error and $R^2$. The coefficient is small and $R^2$ is low. Is the effect of digital technology small, or is the data poor? What would you need to tell the difference?

In [ ]:
model_survey_survey <- lm_robust(
  ln_prod_survey ~ ln_digital_survey + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score,
  data = survey
)

tidy(model_survey_survey) |>
  filter(term == "ln_digital_survey")

In [ ]:
glance(model_survey_survey) |> select(r.squared)

### 2.2 The administrative records arrive: four regressions

> The ministry gives you tax and customs records, `firms_admin.csv`. Merge them by `firm_id` and complete the table.

First, merge and check that all 2,000 firms are matched.

In [ ]:
admin <- read_csv("https://raw.githubusercontent.com/henryredondo/henryredondo.github.io/main/datasets/firms_admin.csv",
                  show_col_types = FALSE)
firms_merged <- inner_join(survey, admin, by = "firm_id")
nrow(firms_merged)

The four models differ only in which measure of $y$ and of $x$ they use. All are estimated on the matched sample; the survey-only model is re-estimated there for comparison.

| Model | $y$ | $x$ |
|:---|:---|:---|
| 1 | survey | survey |
| 2 | survey | records |
| 3 | records | survey |
| 4 | records | records |

In [ ]:
model_survey_survey <- lm_robust(
  ln_prod_survey ~ ln_digital_survey + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score,
  data = firms_merged
)

model_survey_admin <- lm_robust(
  ln_prod_survey ~ ln_digital_admin + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score,
  data = firms_merged
)

model_admin_survey <- lm_robust(
  ln_prod_admin ~ ln_digital_survey + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score,
  data = firms_merged
)

model_admin_admin <- lm_robust(
  ln_prod_admin ~ ln_digital_admin + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score,
  data = firms_merged
)

All four estimates in one table. `coef_map` keeps only digital capital and puts both of its measures on the same row. Robust (HC2) standard errors in parentheses; all models include the controls.

In [ ]:
modelsummary(
  list("1. Survey y, survey x" = model_survey_survey,
       "2. Survey y, admin x"  = model_survey_admin,
       "3. Admin y, survey x"  = model_admin_survey,
       "4. Admin y, admin x"   = model_admin_admin),
  coef_map = c(ln_digital_survey = "Digital capital (log)",
               ln_digital_admin  = "Digital capital (log)"),
  gof_map = gof_rows, fmt = 4
)

**Reading the table:**
- Compare columns **2 and 4** (only $y$ changes) to isolate measurement error in $y$.
- Compare columns **3 and 4** (only $x$ changes) to isolate measurement error in $x$.

### 2.3 Measurement error in the dependent variable

> Compare row 4 with row 2: only $y$ changes. What happens to the coefficient, the standard error and $R^2$? Explain it with $y_{\text{survey}} = y + u$.

We compute the two measurement errors as the difference between the survey and the records.

In [ ]:
firms_merged <- firms_merged |>
  mutate(error_in_y = ln_prod_survey - ln_prod_admin,
         error_in_x = ln_digital_survey - ln_digital_admin)

firms_merged |>
  summarise(sd_error_in_y = sd(error_in_y),
            sd_clean_y = sd(ln_prod_admin),
            correlation_error_with_x = cor(error_in_y, ln_digital_admin))

If the error $u$ is orthogonal to all regressors, it goes into the residual: it does **not bias** the coefficients, but it **increases their sampling variability**.

In [ ]:
tibble(se_with_clean_y = model_admin_admin$std.error[["ln_digital_admin"]],
       se_with_noisy_y = model_survey_admin$std.error[["ln_digital_admin"]]) |>
  mutate(ratio = se_with_noisy_y / se_with_clean_y)

### 2.4 Measurement error in the explanatory variable

> Compare row 4 with row 3: only $x$ changes. By what factor does the coefficient shrink? Compute the reliability ratio and check that it predicts the shrinkage.

In [ ]:
firms_merged |>
  summarise(sd_error_in_x = sd(error_in_x),
            correlation_error_with_x = cor(error_in_x, ln_digital_admin))

With classical measurement error in one regressor, the slope is attenuated by the **reliability ratio**

$$\lambda = \frac{\mathrm{Var}(\text{true } x \mid \text{controls})}{\mathrm{Var}(\text{noisy } x \mid \text{controls})}, \qquad \text{plim}\,\hat\beta_{\text{noisy}} = \lambda\,\beta .$$

To compute it, we residualise each measure of $x$ on the same controls: residual = observed value − fitted value.

In [ ]:
true_x_regression <- lm_robust(
  ln_digital_admin ~ ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score,
  data = firms_merged
)

noisy_x_regression <- lm_robust(
  ln_digital_survey ~ ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score,
  data = firms_merged
)

x_true_after_controls  <- firms_merged$ln_digital_admin  - fitted(true_x_regression)
x_noisy_after_controls <- firms_merged$ln_digital_survey - fitted(noisy_x_regression)

reliability_ratio <- var(x_true_after_controls) / var(x_noisy_after_controls)
reliability_ratio

Does $\lambda \times$ (clean slope) predict the noisy slope?

In [ ]:
predicted_noisy_slope <- reliability_ratio * coef(model_admin_admin)[["ln_digital_admin"]]

tibble(predicted = predicted_noisy_slope,
       estimated = coef(model_admin_survey)[["ln_digital_survey"]])

The attenuation formula is a **population** result; the two sample numbers need not coincide exactly. In this case measurement error attenuates the slope.

### 2.5 Decision

> Which error is more harmful for the ministry's question? Which would you ask the ministry to fix first?

**From now on we use the administrative measures**, renamed `ln_prod` and `ln_digital`.

In [ ]:
firms <- firms_merged |>
  mutate(ln_prod = ln_prod_admin, ln_digital = ln_digital_admin)

---
# Part 3. Building the model (Unit 2)

The outcome is `ln_prod` and the regressor of interest is `ln_digital`. Management quality is still observed: it enters in the last model.

The coefficient on `ln_digital` is a **conditional elasticity**. It is causal only under the appropriate identification assumptions.

### 3.1 Add controls one block at a time

> Estimate six models, adding one block of controls at a time: (1) `ln_digital` alone; (2) + `ln_employees`; (3) + `firm_age` and `firm_age_sq`; (4) + `skilled_share` and `size_x_skill`; (5) + `sector`; (6) + `management_score`.

In [ ]:
nrow(firms)

In [ ]:
model_1 <- lm_robust(
  ln_prod ~ ln_digital,
  data = firms
)

model_2 <- lm_robust(
  ln_prod ~ ln_digital + ln_employees,
  data = firms
)

model_3 <- lm_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq,
  data = firms
)

model_4 <- lm_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill,
  data = firms
)

model_5 <- lm_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms
)

model_6 <- lm_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score,
  data = firms
)

**Notes on the formulas**
- R creates the sector dummies automatically; food is the reference.
- Writing `ln_employees * skilled_share` in a formula would include both main effects and their interaction, which is equivalent to the terms used here.

The table adds a row with the number of regressors besides the constant: the rank of each model minus one (the intercept). `add_rows` needs one column per model, in the same order as the models.

In [ ]:
regressor_row <- tibble(
  term = "Regressors (excl. constant)",
  m1 = as.character(model_1$rank - 1),
  m2 = as.character(model_2$rank - 1),
  m3 = as.character(model_3$rank - 1),
  m4 = as.character(model_4$rank - 1),
  m5 = as.character(model_5$rank - 1),
  m6 = as.character(model_6$rank - 1)
)

modelsummary(
  list("1. Digital only"          = model_1,
       "2. + size"                = model_2,
       "3. + age, age squared"    = model_3,
       "4. + skills, interaction" = model_4,
       "5. + sector"              = model_5,
       "6. + management"          = model_6),
  coef_map = c(ln_digital = "Digital capital (log)"),
  gof_map = gof_rows, fmt = 4,
  add_rows = regressor_row
)

### 3.2 Adjusted $R^2$ by hand (model 6)

> Compute the adjusted $R^2$ of model 6 by hand ($n = 2{,}000$; $k = 10$ regressors besides the constant). Why can $R^2$ never fall when a variable is added, while adjusted $R^2$ can?

$$\bar R^2 = 1 - (1 - R^2)\,\frac{n - 1}{n - k - 1}$$

In [ ]:
n <- glance(model_6)$nobs
k <- model_6$rank - 1
r_squared <- model_6$r.squared

adjusted_by_hand <- 1 - (1 - r_squared) * (n - 1) / (n - k - 1)
c(by_hand = adjusted_by_hand, from_lm_robust = model_6$adj.r.squared)

Adding regressors cannot reduce $R^2$ when the sample is unchanged. Adjusted $R^2$ penalises the number of parameters and can decrease.

### 3.3 Why does the coefficient on digital capital fall?

> The coefficient on `ln_digital` falls a lot across the six columns. Which block moves it most? Why do the controls change it at all?

Digitally intensive firms also differ in size, skills, sector and management. Compare the conditional associations across the six specifications. For instance, the correlation between management quality and digital adoption:

In [ ]:
cor(firms$management_score, firms$ln_digital)

### 3.4 The quadratic in firm age

> In model 6, compute the effect of one more year of age at ages 5, 20 and 40, and the age at which the effect is zero. Interpret.

In [ ]:
summary(model_6)

The marginal effect of age on log productivity is

$$\frac{\partial\, \ln \text{prod}}{\partial\, \text{age}} = b_{\text{age}} + 2\, b_{\text{age}^2}\, \text{age}.$$

In [ ]:
b <- coef(model_6)

tibble(age = c(5, 20, 40)) |>
  mutate(marginal_effect = b[["firm_age"]] + 2 * b[["firm_age_sq"]] * age)

Setting the marginal effect to zero gives the turning point, $\text{age}^* = -b_{\text{age}} / (2\, b_{\text{age}^2})$. It is a maximum when the squared coefficient is negative.

In [ ]:
age_of_maximum <- -b[["firm_age"]] / (2 * b[["firm_age_sq"]])
age_of_maximum

### 3.5 The interaction between firm size and skilled share

> Compute the effect of `ln_employees` at the 10th percentile, the mean and the 90th percentile of `skilled_share`. Why can the coefficient on `ln_employees` not be read on its own?

The size elasticity depends on the skilled share:

$$\frac{\partial\, \ln \text{prod}}{\partial\, \ln \text{employees}} = b_{\text{size}} + b_{\text{size}\times\text{skill}}\; \text{skilled share}.$$

In [ ]:
share_quantiles <- quantile(firms$skilled_share, c(0.10, 0.90), names = FALSE)

tibble(where = c("10th percentile", "Mean", "90th percentile"),
       skilled_share = c(share_quantiles[1], mean(firms$skilled_share), share_quantiles[2])) |>
  mutate(effect_of_size = b[["ln_employees"]] + b[["size_x_skill"]] * skilled_share)

The coefficient on `ln_employees` alone is the elasticity at `skilled_share = 0`.

### 3.6 Overfitting: add 20 irrelevant variables

> Add 20 variables of pure noise (`rnorm`) to model 6. Compare the gains in $R^2$ and adjusted $R^2$. Test the 20 variables jointly. What do you conclude?

In [ ]:
set.seed(20261123)
firms_with_noise <- firms
for (i in 1:20) {
  firms_with_noise[[sprintf("noise_%02d", i)]] <- rnorm(nrow(firms))
}

`update()` takes the formula of model 6 and adds the 20 noise variables explicitly.

In [ ]:
model_6_plus_noise <- update(model_6,
  . ~ . + noise_01 + noise_02 + noise_03 + noise_04 + noise_05 +
    noise_06 + noise_07 + noise_08 + noise_09 + noise_10 +
    noise_11 + noise_12 + noise_13 + noise_14 + noise_15 +
    noise_16 + noise_17 + noise_18 + noise_19 + noise_20,
  data = firms_with_noise
)

In [ ]:
bind_rows(model_6 = glance(model_6),
          model_6_plus_noise = glance(model_6_plus_noise), .id = "model") |>
  select(model, r.squared, adj.r.squared) |>
  mutate(gain_in_r_squared = r.squared - lag(r.squared),
         gain_in_adj_r_squared = adj.r.squared - lag(adj.r.squared))

The digital coefficient and its robust standard error, with and without the noise:

In [ ]:
modelsummary(
  list("Model 6" = model_6, "Model 6 + 20 noise variables" = model_6_plus_noise),
  coef_map = c(ln_digital = "Digital capital (log)"),
  gof_map = gof_rows, fmt = 4
)

**Classical $F$-test of the 20 additional coefficients.** Model 6 is the *restricted* model (the 20 coefficients set to zero) and model 6 + noise the *unrestricted* one. With $q$ restrictions and $n - k_u - 1$ residual degrees of freedom in the unrestricted model:

$$F = \frac{(R^2_u - R^2_r)/q}{(1 - R^2_u)/(n - k_u - 1)}$$

In [ ]:
q_noise <- 20
df_residual_noise <- nrow(firms) - model_6_plus_noise$rank   # n - k_u - 1
r2_restricted <- model_6$r.squared
r2_unrestricted <- model_6_plus_noise$r.squared

F_noise <- ((r2_unrestricted - r2_restricted) / q_noise) /
  ((1 - r2_unrestricted) / df_residual_noise)

c(F = F_noise, p_value = pf(F_noise, q_noise, df_residual_noise, lower.tail = FALSE))

> **Rule to remember.** Adjusted $R^2$ increases exactly when the classical partial $F$-statistic exceeds 1.

### 3.7 The rank condition

> Add the three headcounts to model 6: `n_workers_skilled`, `n_workers_unskilled` and `n_workers_total`. (a) What does R report? (b) Check the rank of the regressor matrix and try to invert $X'X$. (c) Drop the total, then drop the unskilled instead. Compare $R^2$, the coefficient on `ln_digital` and the coefficient on `n_workers_skilled`. (d) Which question can the data not answer?

Total workers = skilled workers + unskilled workers, for every firm:

In [ ]:
all(firms$n_workers_total == firms$n_workers_skilled + firms$n_workers_unskilled)

**(a)** Including all three headcounts:

In [ ]:
model_collinear <- lm_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score +
    n_workers_skilled + n_workers_unskilled + n_workers_total,
  data = firms
)

coef(model_collinear)[c("n_workers_skilled", "n_workers_unskilled", "n_workers_total")]

`lm_robust()` reports `NA` for one redundant coefficient. It uses a column-pivoting QR decomposition, so it may drop a different headcount than `lm()` would.

**(b)** $X$ has more columns than its rank, so $X'X$ is singular. `try()` lets the notebook continue after the expected error.

In [ ]:
X <- model.matrix(
  ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score +
    n_workers_skilled + n_workers_unskilled + n_workers_total,
  data = firms
)

c(columns_in_X = ncol(X), rank_of_X = qr(X)$rank)

In [ ]:
try(solve(t(X) %*% X))  # Expected to fail

**(c)** Drop one of the redundant headcounts, in two different ways.

In [ ]:
model_drop_total <- lm_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score +
    n_workers_skilled + n_workers_unskilled,
  data = firms
)

model_drop_unskilled <- lm_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score +
    n_workers_skilled + n_workers_total,
  data = firms
)

In [ ]:
bind_rows(drop_total = glance(model_drop_total),
          drop_unskilled = glance(model_drop_unskilled), .id = "model") |>
  select(model, r.squared)

In [ ]:
bind_rows(drop_total = tidy(model_drop_total),
          drop_unskilled = tidy(model_drop_unskilled), .id = "model") |>
  filter(term %in% c("ln_digital", "n_workers_skilled")) |>
  select(model, term, estimate) |>
  pivot_wider(names_from = term, values_from = estimate)

The fit and the digital coefficient are unchanged. The headcount coefficients have **different interpretations** because the variables held fixed differ.

---
# Part 4. Omitted variable bias (Unit 2)

In Part 3 we had a good measure of management quality. Suppose we did not. What would the ministry have learned?

- The **short** regression leaves out `management_score`.
- The **long** regression includes it.

### 4.1 Predict the sign

> Before estimating: in which direction do you expect the bias from omitting management? State the two facts that determine it.

Better-managed firms are more productive **and** adopt more digital capital. If both conditional relationships are positive, omitting management biases the digital coefficient **upwards**, under the maintained model.

### 4.2 Short and long regressions

> Estimate the short and the long regression. Report both coefficients on `ln_digital`, the bias, and the bias as a percentage of the long coefficient.

In [ ]:
short_regression <- lm_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms
)

long_regression <- lm_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector + management_score,
  data = firms
)

In [ ]:
modelsummary(
  list(Short = short_regression, Long = long_regression),
  coef_map = c(ln_digital = "Digital capital (log)",
               management_score = "Management score"),
  gof_map = gof_rows, fmt = 4
)

In [ ]:
short_slope <- coef(short_regression)[["ln_digital"]]
long_slope  <- coef(long_regression)[["ln_digital"]]

tibble(difference = short_slope - long_slope,
       difference_percent = 100 * (short_slope - long_slope) / long_slope)

### 4.3 The omitted-variable-bias formula

> Compute $g$ and $d$. Verify that short = long + $g\,d$. Say in words what $g$ and $d$ mean.

$$\hat\beta_{\text{short}} = \hat\beta_{\text{long}} + g \cdot d$$

- $g$: coefficient on management in the **long** regression.
- $d$: coefficient on digital capital in the **auxiliary** regression of management on all the regressors of the short regression.

In [ ]:
g <- coef(long_regression)[["management_score"]]

auxiliary_regression <- lm_robust(
  management_score ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms
)

d <- coef(auxiliary_regression)[["ln_digital"]]

c(g = g, d = d, g_times_d = g * d)

The identity holds **exactly** in a common estimation sample (the difference is rounding error):

In [ ]:
tibble(short_slope = short_slope, long_plus_g_times_d = long_slope + g * d) |>
  mutate(difference = short_slope - long_plus_g_times_d)

### 4.4 Implications for the ministry

> A colleague prefers the short regression because its standard error is smaller. Reply.

Compare the short estimate (about 0.21) with the long estimate (about 0.12). The short one overstates the return if the long model is correctly specified. In real applications management quality may be **unobserved**, which motivates Part 5.

---
# Part 5. Instrumental variables (Unit 3)

Management quality is now **unavailable**: the ministry cannot measure it, so we drop `management_score`.

A valid instrument $Z$ must be:
- **Relevant:** it moves digital capital.
- **Exogenous / excluded:** it is orthogonal to the structural error, and has no effect on productivity except through digital capital.

A causal interpretation also requires the maintained structural model.

| Candidate | Description |
|:---|:---|
| `grant_lottery` | Randomly assigned grant |
| `ln_dist_hub` | Log distance to the nearest technology hub |
| `info_mailing` | Randomly assigned information leaflet |
| `neighbour_digital` | Average digital intensity of neighbouring firms |

### 5.1 OLS when management becomes unobserved

> Why can we not simply control for management? What does OLS give without it? Which of the two instrument conditions can be tested with the data?

We keep `firms` intact for the earlier parts and use a separate dataset, `firms_iv`, without management.

In [ ]:
benchmark_with_management <- long_slope
firms_iv <- select(firms, -management_score)

ols_short <- lm_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

tidy(ols_short) |>
  filter(term == "ln_digital")

In [ ]:
benchmark_with_management

### 5.2 Instrument relevance: first-stage regressions

> Complete the table for the four candidates: first-stage $F$, why it might be excluded, and a story that breaks it.

First the **restricted** first stage, with the controls only (no excluded instruments):

In [ ]:
first_stage_controls <- lm_robust(
  ln_digital ~ ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

Then one **unrestricted** first stage per candidate (and two pairs):

In [ ]:
first_stage <- lm_robust(
  ln_digital ~ grant_lottery + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

first_stage_hub <- lm_robust(
  ln_digital ~ ln_dist_hub + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

first_stage_mailing <- lm_robust(
  ln_digital ~ info_mailing + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

first_stage_neighbours <- lm_robust(
  ln_digital ~ neighbour_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

first_stage_grant_hub <- lm_robust(
  ln_digital ~ grant_lottery + ln_dist_hub + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

first_stage_grant_neighbours <- lm_robust(
  ln_digital ~ grant_lottery + neighbour_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

The first-stage $F$ compares each unrestricted first stage with the restricted one, with the same formula as in 3.6. Here $q$ is the number of excluded instruments:

$$F = \frac{(R^2_u - R^2_r)/q}{(1 - R^2_u)/(n - k_u - 1)}$$

In [ ]:
first_stage_results <- tibble(
  instruments = c("Grant", "Hub", "Mailing", "Neighbours", "Grant + hub", "Grant + neighbours"),
  r2_unrestricted = c(first_stage$r.squared,
                      first_stage_hub$r.squared,
                      first_stage_mailing$r.squared,
                      first_stage_neighbours$r.squared,
                      first_stage_grant_hub$r.squared,
                      first_stage_grant_neighbours$r.squared),
  q = c(1, 1, 1, 1, 2, 2),                                       # excluded instruments
  df_residual = nrow(firms_iv) - first_stage_controls$rank - q   # n - k_u - 1
) |>
  mutate(r2_restricted = first_stage_controls$r.squared,
         first_stage_F = ((r2_unrestricted - r2_restricted) / q) /
           ((1 - r2_unrestricted) / df_residual))

first_stage_results |> select(instruments, q, first_stage_F)

$F > 10$ is a rough heuristic for the classical first-stage statistic, **not** a general guarantee against weak identification.

### 5.3 Exclusion: what can and cannot be checked

> The grant is a lottery. Regress `grant_lottery` on the controls and test them jointly. What does the result show, and what can it not show?

With `se_type = "classical"`, `glance()` reports the classical overall $F$-test. Without `se_type` (the HC2 default) the same code would give a robust Wald version.

In [ ]:
balance_check <- lm_robust(
  grant_lottery ~ ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv, se_type = "classical"
)

glance(balance_check) |>
  select(statistic, p.value, df.residual) |>
  mutate(df = balance_check$rank - 1, .before = df.residual)

**What a balance test can and cannot show**
- A large $p$-value does **not** prove balance or exclusion.
- The grant is randomised, but it could affect productivity through other inputs.
- Distance to the hub may capture local economic conditions.
- Neighbouring firms may share unobserved management practices.

### 5.4 The IV estimator as a ratio

> With the grant alone, compute the reduced form, the first stage and their ratio. Compare the ratio with `iv_robust()`. Interpret all three numbers.

With one excluded instrument and one endogenous regressor:

$$\hat\beta_{IV} = \frac{\text{reduced-form coefficient}}{\text{first-stage coefficient}}$$

In [ ]:
reduced_form <- lm_robust(
  ln_prod ~ grant_lottery + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

effect_of_grant_on_productivity <- coef(reduced_form)[["grant_lottery"]]
effect_of_grant_on_digital      <- coef(first_stage)[["grant_lottery"]]

tibble(reduced_form = effect_of_grant_on_productivity,
       first_stage = effect_of_grant_on_digital) |>
  mutate(wald_ratio = reduced_form / first_stage)

**`iv_robust()` syntax**

```r
iv_robust(y ~ endogenous + controls | excluded instruments + controls, ...)
```

Exogenous controls must appear on **both** sides of the vertical bar.

In [ ]:
iv_grant <- iv_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector |
    grant_lottery + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

tidy(iv_grant) |>
  filter(term == "ln_digital")

### 5.5 Two-stage least squares by hand

> Regress `ln_prod` on the fitted values of `ln_digital`. Compare the standard error with the one from `iv_robust()`. Why do they differ?

In [ ]:
firms_iv <- mutate(firms_iv, ln_digital_predicted = fitted(first_stage))

second_stage_by_hand <- lm_robust(
  ln_prod ~ ln_digital_predicted + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

c(second_stage_by_hand = coef(second_stage_by_hand)[["ln_digital_predicted"]],
  iv_robust = coef(iv_grant)[["ln_digital"]])

The point estimates agree, but the second-stage OLS residuals are **incorrect** for inference on the structural equation. Both use HC2.

In [ ]:
c(se_second_stage_by_hand = second_stage_by_hand$std.error[["ln_digital_predicted"]],
  se_correct_iv_robust = iv_grant$std.error[["ln_digital"]])

### 5.6 Two valid instruments: precision and overidentification

> Use the grant and the hub together. Report the estimate and standard error. Compute the $J$ statistic by hand. Interpret it. What can $J$ not detect?

`diagnostics = TRUE` adds a first-stage test, an endogeneity test and an overidentification test.

In [ ]:
iv_grant_and_hub <- iv_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector |
    grant_lottery + ln_dist_hub + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv, diagnostics = TRUE
)

2SLS with robust (HC2) standard errors in parentheses:

In [ ]:
modelsummary(
  list("Grant only" = iv_grant, "Grant + hub" = iv_grant_and_hub),
  coef_map = c(ln_digital = "Digital capital (log)"),
  gof_map = gof_rows[1, ], fmt = 4   # only N: R2 is not meaningful for IV
)

**Classical Sargan test by hand**

1. Compute the structural 2SLS residual. The fitted values of `iv_robust()` use the actual `ln_digital`, so $y - \text{fitted}$ is the structural residual.
2. Regress it on **all** instruments (excluded instruments and controls).
3. $J = n R^2 \sim \chi^2_1$: one overidentifying restriction (two excluded instruments, one endogenous $x$).

The $\chi^2$ reference distribution requires homoskedasticity and the usual regularity conditions. This case allows heteroskedasticity, so treat the classical Sargan $p$-value as **illustrative**, not robust evidence.

In [ ]:
firms_iv <- mutate(firms_iv, iv_residual = ln_prod - fitted(iv_grant_and_hub))

residual_regression <- lm_robust(
  iv_residual ~ grant_lottery + ln_dist_hub + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv, se_type = "classical"
)

sargan_statistic <- glance(residual_regression)$nobs * residual_regression$r.squared
tibble(Sargan = sargan_statistic,
       p_value = pchisq(sargan_statistic, df = 1, lower.tail = FALSE))

**Built-in diagnostics.** The three tests depend on `se_type`:

| Test | `se_type = "classical"` | default (HC2) |
|:---|:---|:---|
| First-stage $F$ (`weakinst`) | classical | heteroskedasticity-robust |
| Wu–Hausman (`endogeneity`) | classical | heteroskedasticity-robust |
| Overidentification (`overid`) | Sargan | Wooldridge robust score test |

`update()` re-estimates the same model with classical standard errors so that both versions can be compared.

In [ ]:
iv_grant_and_hub_classical <- update(iv_grant_and_hub, se_type = "classical")

bind_rows(classical = glance(iv_grant_and_hub_classical),
          robust_HC2 = glance(iv_grant_and_hub), .id = "version") |>
  select(version, statistic.weakinst, statistic.endogeneity, p.value.endogeneity,
         statistic.overid, p.value.overid)

A non-rejection does **not** prove instrument validity.

### 5.7 A weak instrument: the information mailing

> Estimate with the mailing alone. Report the estimate, the standard error and a 95% interval. What do we learn?

In [ ]:
iv_mailing <- iv_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector |
    info_mailing + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

tidy(iv_mailing) |>
  filter(term == "ln_digital") |>
  select(estimate, std.error, conf.low, conf.high)

- `conf.low` and `conf.high` come from `estimatr` (t distribution). With 2,000 firms they are practically the same as estimate ± 1.96 standard errors.
- The classical first-stage $F$ is about 1.5. Weak instruments can cause severe finite-sample distortions and unreliable conventional confidence intervals.
- The interval above is **illustrative**, not weak-instrument-robust inference.

### 5.8 A strong but invalid instrument: neighbours

> Estimate with the neighbours' adoption alone and compare with OLS. Then use the grant and the neighbours together and run the $J$ test. Why can a strong instrument be worse than none?

In [ ]:
iv_neighbours <- iv_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector |
    neighbour_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

tidy(iv_neighbours) |>
  filter(term == "ln_digital")

A strong first stage does **not** establish instrument exogeneity: neighbours may share unobserved management characteristics with the firm. Compare the IV estimate with the benchmark and with OLS.

Now the grant and the neighbours together:

In [ ]:
iv_grant_and_neighbours <- iv_robust(
  ln_prod ~ ln_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector |
    grant_lottery + neighbour_digital + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv, diagnostics = TRUE
)

iv_grant_and_neighbours_classical <- update(iv_grant_and_neighbours, se_type = "classical")

bind_rows(classical = glance(iv_grant_and_neighbours_classical),
          robust_HC2 = glance(iv_grant_and_neighbours), .id = "version") |>
  select(version, statistic.weakinst, statistic.endogeneity, p.value.endogeneity,
         statistic.overid, p.value.overid)

The overidentification test can detect **disagreement** between instruments, but cannot identify which instrument is invalid. Similar violations may go undetected.

### Summary of the estimates

Each column is a model already estimated above. The extra rows hold the **classical** first-stage $F$ and Sargan $p$-value. Robust (HC2) standard errors in parentheses.

First the extra rows, one column per model in the order of the table (empty where a statistic does not apply):

In [ ]:
F_values <- sprintf("%.1f", first_stage_results$first_stage_F)
names(F_values) <- first_stage_results$instruments

sargan_hub        <- sprintf("%.3f", glance(iv_grant_and_hub_classical)$p.value.overid)
sargan_neighbours <- sprintf("%.3f", glance(iv_grant_and_neighbours_classical)$p.value.overid)

diagnostic_rows <- tribble(
  ~term, ~ols_short, ~ols_long, ~grant, ~grant_hub, ~mailing, ~neighbours, ~grant_neighbours,
  "First-stage F (classical)", "", "", F_values[["Grant"]], F_values[["Grant + hub"]],
    F_values[["Mailing"]], F_values[["Neighbours"]], F_values[["Grant + neighbours"]],
  "Sargan p-value (classical)", "", "", "", sargan_hub, "", "", sargan_neighbours
)

In [ ]:
modelsummary(
  list("OLS, no management"       = ols_short,
       "OLS with management"      = long_regression,
       "IV: grant"                = iv_grant,
       "IV: grant + hub"          = iv_grant_and_hub,
       "IV: mailing (weak)"       = iv_mailing,
       "IV: neighbours (invalid)" = iv_neighbours,
       "IV: grant + neighbours"   = iv_grant_and_neighbours),
  coef_map = c(ln_digital = "Digital capital (log)"),
  gof_map = gof_rows[1, ], fmt = 4,
  add_rows = diagnostic_rows
)

### 5.9 Extension: instrumental variables and measurement error

> Use the *survey* measure `ln_digital_survey`, with the grant as instrument and without management. What does IV fix here besides the omitted variable? What does it cost?

If the lottery is orthogonal to **both** the structural error and the measurement error, IV can address both sources of endogeneity.

In [ ]:
ols_noisy_x <- lm_robust(
  ln_prod ~ ln_digital_survey + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

iv_noisy_x <- iv_robust(
  ln_prod ~ ln_digital_survey + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector |
    grant_lottery + ln_employees + firm_age + firm_age_sq +
    skilled_share + size_x_skill + sector,
  data = firms_iv
)

modelsummary(
  list(OLS = ols_noisy_x, IV = iv_noisy_x),
  coef_map = c(ln_digital_survey = "Digital capital, survey (log)"),
  gof_map = gof_rows[1, ], fmt = 4
)

- The noisy OLS estimate can be close to the benchmark **by coincidence**: attenuation and omitted-variable bias may offset one another.
- IV may be much less precise because the noisy measure has more residual variation.

### 5.10 Policy implications

> **Memo.** Write five or six lines for the ministry: the elasticity with a 95% interval; how much of the OLS number is bias; what the grant does to digital capital and to productivity; which instruments you trust; two things that could still be wrong.

**(a) The elasticity:** 2SLS using the grant and distance to the hub.

In [ ]:
elasticity <- tidy(iv_grant_and_hub) |>
  filter(term == "ln_digital") |>
  select(estimate, std.error, conf.low, conf.high)
elasticity

The interval is wide. The classical Wu–Hausman test also has limited evidence against exogeneity here. The OVB argument motivates IV; whether the OLS point estimate falls inside an IV interval is **not** a formal test.

**(b) The effect of grant assignment:** reduced form and first stage.
- Random assignment identifies **intention-to-treat** effects under the design.
- Exponentiating a log difference gives a proportional difference in **geometric** means (conditional on the controls), not necessarily in arithmetic means.

In [ ]:
tibble(winners_digital_capital_percent_higher = 100 * (exp(effect_of_grant_on_digital) - 1),
       winners_productivity_percent_higher = 100 * (exp(effect_of_grant_on_productivity) - 1))

**(c) Recommendation.** The benchmark and IV estimates suggest a return of about 0.12–0.13, rather than 0.21. A grant raises digital capital by roughly 37% and productivity by roughly 4% in the simulated case. A cost-benefit analysis is needed to assess whether the policy is worthwhile.